In [ ]:
import numpy as np
import pandas as pd
from pynq import Overlay, allocate
import time

In [ ]:
# Software convolution for verification
def conv2d_sw(image, kernel, norm_shift):
    H, W = image.shape
    K = kernel.shape[0]

    out_h = H - K + 1
    out_w = W - K + 1

    out = np.zeros((out_h, out_w), dtype=np.uint8)

    for i in range(out_h):
        for j in range(out_w):
            window = image[i:i+K, j:j+K]
            s = int(np.sum(window * kernel))

            # match HLS
            bias = (1 << (norm_shift - 1)) if norm_shift > 0 else 0
            result = (s + bias) >> norm_shift

            # CLAMP
            result = max(0, min(255, result))

            out[i, j] = result
    
    return out

In [ ]:
# Initialize overlay and IPs
overlay = Overlay("../vivado/fpga_2d_convolution.bit")
dma = overlay.axi_dma
hls_ip = overlay.conv_dataflow_stream

In [ ]:
# Helper functions to pack/unpack 8-bit pixels into 32-bit words for DMA transfer
def pack_pixels(image):
    flat = image.flatten()
    packed = np.zeros(( (len(flat)+3)//4, ), dtype=np.uint32)

    word = 0
    count = 0
    idx = 0

    for px in flat:
        word |= int(px) << (8 * count)
        count += 1
        
        if count == 4:
            packed[idx] = word
            idx += 1
            word = 0
            count = 0

    if count != 0:
        packed[idx] = word

    return packed

def unpack_pixels(packed, total_pixels):
    out = np.zeros(total_pixels, dtype=np.uint8)
    idx = 0

    finished = False
    for word in packed:
        for i in range(4):
            if idx < total_pixels:
                out[idx] = (word >> (8 * i)) & 0xFF
                idx += 1
            
            if idx >= total_pixels:
                finished = True
                break
            
        if finished:
            break

    return out

def pack_pixels_vectorized(image):
    flat = np.ascontiguousarray(image.flatten(), dtype=np.uint8)
    
    pad = (-len(flat)) % 4
    if pad:
        flat = np.pad(flat, (0, pad))

    # SAFE: group into 4 bytes explicitly
    flat = flat.reshape(-1, 4)

    # reinterpret as uint32
    packed = flat.view('<u4').reshape(-1)

    return packed


def unpack_pixels_vectorized(packed, total_pixels):
    # safe reinterpretation
    flat = packed.view('<u1')
    return flat[:total_pixels]

In [ ]:
def conv2d_hw(image, kernel, norm_shift):
    flat_kernel = kernel.flatten()

    # 1. Write kernel
    for i in range(len(flat_kernel)):
        hls_ip.write(0x10 + 0x8*i, int(flat_kernel[i]))

    # 2. Write params
    H, W = image.shape
    hls_ip.write(0x58, H)
    hls_ip.write(0x60, W)
    hls_ip.write(0x68, norm_shift)

    # ===== TIMING START =====
    t0 = time.time()

    # 3. Pack input
    t_pack_start = time.time()
    packed_input = pack_pixels_vectorized(image)

    input_buffer = allocate(shape=packed_input.shape, dtype=np.uint32)
    input_buffer[:] = packed_input
    t_pack_end = time.time()

    # 4. Allocate output buffer
    out_h = H - 2
    out_w = W - 2
    total_pixels = out_h * out_w
    packed_out_size = (total_pixels + 3) // 4

    output_buffer = allocate(shape=(packed_out_size,), dtype=np.uint32)

    try:
        # 5. Start IP
        hls_ip.write(0x00, 1)

        # 6. DMA + compute
        t_hw_start = time.time()

        dma.sendchannel.transfer(input_buffer) 
        dma.recvchannel.transfer(output_buffer)

        dma.sendchannel.wait()
        dma.recvchannel.wait()

        t_hw_end = time.time()

        # 7. Unpack output
        t_unpack_start = time.time()
        output = unpack_pixels_vectorized(output_buffer, total_pixels)
        t_unpack_end = time.time()

        t1 = time.time()

    finally:
        input_buffer.freebuffer()
        output_buffer.freebuffer()

    return {
        "output": output.reshape(out_h, out_w),
        "total_time": t1 - t0,
        "hw_time": t_hw_end - t_hw_start,
        "pack_time": t_pack_end - t_pack_start,
        "unpack_time": t_unpack_end - t_unpack_start
    }

In [ ]:
# Tiling approach to handle larger images than the IP can process in one go
def tile_image(image, kernel, norm_shift, tile_size):
    halo = kernel.shape[0] // 2
    padded = np.pad(image, pad_width=halo, mode='constant', constant_values=0)

    H, W = image.shape
    K = kernel.shape[0]

    out_H, out_W = H - K + 1, W - K + 1
    output = np.zeros((out_H, out_W), dtype=np.uint8)

    total_hw = total_pack = total_unpack = 0
    tile_count = 0

    for i in range(0, H, tile_size):
        for j in range(0, W, tile_size):

            # extract tile INCLUDING halo
            tile = padded[
                i : i + tile_size + 2*halo,
                j : j + tile_size + 2*halo
            ]

            out = conv2d_hw(tile, kernel, norm_shift)

            total_hw += out["hw_time"]
            total_pack += out["pack_time"]
            total_unpack += out["unpack_time"]
            tile_count += 1

            out_tile = out["output"]
            assert out_tile.shape == (tile.shape[0] - K + 1, tile.shape[1] - K + 1)

            # remove halo
            valid = out_tile

            # placement
            oi_start = i
            oj_start = j

            oi_end = min(i + valid.shape[0], out_H)
            oj_end = min(j + valid.shape[1], out_W)

            output[oi_start:oi_end, oj_start:oj_end] = valid[
                :oi_end - oi_start,
                :oj_end - oj_start
            ]

    return {
        "output": output,
        "tiles": tile_count,
        "hw_time": total_hw,
        "pack_time": total_pack,
        "unpack_time": total_unpack
    }

In [ ]:
# Kernels

# Sobel X, Sobel Y, and Gaussian kernels
sobel_x = np.array([
    [-1, 0, 1],
    [-2, 0, 2],
    [-1, 0, 1]
], dtype=np.int32)

sobel_y = np.array([
    [-1, -2, -1],
    [ 0,  0,  0],
    [ 1,  2,  1]
], dtype=np.int32)

gaussian = np.array([
    [1, 2, 1],
    [2, 4, 2],
    [1, 2, 1]
], dtype=np.int32)

In [ ]:
# Benchmarking
sizes = [2**i for i in range(2, 13)]  # 4 -> 4096
tile_sizes = [2**i for i in range(5, 10)]  # 32 -> 512

rows = [] # Store results for DataFrame
tile_rows = [] # Store tile results for DataFrame
verbose = False
square_only = True
def benchmark(kernel, norm_shift, label):
    sw_times = []
    hw_times = []

    if verbose:
        print(f"{label} Kernel Benchmarking:")

    for height in sizes:
        for width in sizes:
            if square_only and height != width:
                continue
            if verbose:
                print(f"\tImage Size: {height}x{width}")

            image = np.random.randint(0, 256, (height, width), dtype=np.uint8)

            # --- Software ---
            out_sw = None
            sw_time = None
            if max(height, width) <= 512:
                t0 = time.time()
                out_sw = conv2d_sw(image, kernel, norm_shift)
                t1 = time.time()
                sw_time = t1 - t0

            # --- Hardware ---
            result = conv2d_hw(image, kernel, norm_shift)
            out_hw = result["output"]
            total_hw_time = result["total_time"]

            # correctness check
            if out_sw is not None:
                assert np.array_equal(out_sw, out_hw), f"\t\tHW VS SW Mismatch at size {height}x{width}"
            if verbose:
                if out_sw is not None:
                    print(f"\t\tHW VS SW Output {height}x{width}: OK")
                else:
                    print(f"\t\tHW Output {height}x{width}: OK (SW skipped)")

            tile_results = {}
            if verbose:
                print(f"\t\tTiled Execution:")

            res_tile = None
            t0_tile = t1_tile = None
            for tile in tile_sizes:
                if tile > max(height, width):
                    continue

                t0_tile = time.time()
                res_tile = tile_image(image, kernel, norm_shift, tile)
                t1_tile = time.time()
                tile_total_time = t1_tile - t0_tile
                tile_results[tile] = {
                    "total_time": tile_total_time,
                    "hw_time": res_tile["hw_time"],
                    "pack_time": res_tile["pack_time"],
                    "unpack_time": res_tile["unpack_time"],
                    "tiles": res_tile["tiles"]
                }

                tile_rows.append({
                    "kernel": label,
                    "height": height,
                    "width": width,
                    "tile_size": tile,
                    "total_time": tile_total_time,
                    "hw_time": res_tile["hw_time"],
                    "pack_time": res_tile["pack_time"],
                    "unpack_time": res_tile["unpack_time"],
                    "tiles": res_tile["tiles"],
                    "speedup_vs_hw": total_hw_time / tile_total_time if tile_total_time > 0 else None,
                    "speedup_vs_sw": None if sw_time is None else (sw_time / tile_total_time),
                    "overhead_time": tile_total_time - res_tile["hw_time"],
                    "compute_ratio": res_tile["hw_time"] / tile_total_time if tile_total_time > 0 else None,
                })

                # correctness check
                assert np.array_equal(out_hw, res_tile["output"]), f"\t\t\tTILE {tile} mismatch\n"
                if verbose:
                    print(f"\t\t\tTILE {tile} Output: OK")

            best_tile = None
            if tile_results:
                best_tile = min(tile_results.items(), key=lambda x: x[1]["total_time"])[0]

            rows.append({
                "kernel": label,
                "height": height,
                "width": width,
                "sw_time": sw_time,
                "hw_total_time": total_hw_time,
                "hw_time": result["hw_time"],
                "pack_time": result["pack_time"],
                "unpack_time": result["unpack_time"],
                "hw_speedup_vs_sw": None if sw_time is None or total_hw_time is None else (sw_time / total_hw_time),
                "overhead_time": result["pack_time"] + result["unpack_time"],
                "best_tile": best_tile,
                "num_tiles": tile_results[best_tile]["tiles"] if best_tile else None,
                "compute_ratio": result["hw_time"] / total_hw_time if total_hw_time > 0 else None
            })

            sw_times.append(sw_time)
            hw_times.append(result["total_time"])
            if verbose:
                print()

    return sw_times, hw_times

In [ ]:
# Run all Kernels
results = {}

results["Sobel X"] = benchmark(sobel_x, norm_shift=0, label="Sobel X")
results["Sobel Y"] = benchmark(sobel_y, norm_shift=0, label="Sobel Y")
results["Gaussian"] = benchmark(gaussian, norm_shift=4, label="Gaussian")

In [ ]:
import pandas as pd
from IPython.display import display

# =========================
# 1. Base DataFrame (SW vs HW)
# =========================
df = pd.DataFrame(rows)

# Only derive what is NOT stored
df["overhead_ratio"] = df["overhead_time"] / df["hw_total_time"]

# =========================
# 2. Tile DataFrame
# =========================
tile_df = pd.DataFrame(tile_rows)

# Only derive missing metrics
tile_df["overhead_ratio"] = tile_df["overhead_time"] / tile_df["total_time"]

df = df.sort_values(["height", "width"])
tile_df = tile_df.sort_values(["height", "width", "tile_size"])

# =========================
# 3. ANALYSIS
# =========================

# ---- HW vs SW ----
print("=== HW vs SW (speedup > 1) ===")
display(df[df["hw_speedup_vs_sw"] > 1])

# ---- Average HW overhead ----
print("\n=== Average HW overhead ratio per kernel ===")
display(df.groupby("kernel")[["overhead_ratio"]].mean())

# ---- Compute ratio (VERY IMPORTANT) ----
print("\n=== HW Compute Ratio (how much is real compute) ===")
display(df.groupby("kernel")[["compute_ratio"]].mean())

# =========================
# TILE ANALYSIS
# =========================

# ---- Best tile per image ----
print("\n=== Best tile size per image ===")
display(df[["height", "width", "best_tile", "num_tiles"]])

# ---- Most common best tile ----
print("\n=== Most frequent best tile ===")
display(df["best_tile"].value_counts())

# ---- Average tile performance ----
print("\n=== Average time per tile size ===")
display(tile_df.groupby("tile_size")[["total_time"]].mean())

# ---- Speedup vs HW ----
print("\n=== Tile speedup vs HW ===")
display(tile_df.groupby("tile_size")[["speedup_vs_hw"]].mean())

# ---- Speedup vs SW ----
print("\n=== Tile speedup vs SW (valid only) ===")
display(
    tile_df[tile_df["speedup_vs_sw"].notna()]
    .groupby("tile_size")[["speedup_vs_sw"]]
    .mean()
)

# ---- Overhead analysis ----
print("\n=== Tile overhead ratio ===")
display(tile_df.groupby("tile_size")[["overhead_ratio"]].mean())

# ---- Compute ratio ----
print("\n=== Tile compute ratio (how much is actual FPGA work) ===")
display(tile_df.groupby("tile_size")[["compute_ratio"]].mean())

# =========================
# 4. ADVANCED INSIGHT
# =========================

# Merge df with tile_df to get best tile performance
best_tile_perf = df.merge(
    tile_df.drop_duplicates(subset=["height", "width", "tile_size"]),
    left_on=["height", "width", "best_tile"],
    right_on=["height", "width", "tile_size"],
    how="left"
)

# Compute improvement vs HW
best_tile_perf["best_tile_speedup_vs_hw"] = (
    best_tile_perf["hw_total_time"] / best_tile_perf["total_time"]
)

print("\n=== Best tile improvement vs HW ===")
display(best_tile_perf[
    ["height", "width", "best_tile", "best_tile_speedup_vs_hw"]
])

print("\n=== Average improvement of best tile ===")
display(best_tile_perf[["best_tile_speedup_vs_hw"]].mean())

best_tile_perf["best_tile_speedup_vs_sw"] = (
    best_tile_perf["sw_time"] / best_tile_perf["total_time"]
)

print("\n=== Best tile improvement vs SW ===")
display(
    best_tile_perf[best_tile_perf["best_tile_speedup_vs_sw"].notna()][
        ["height", "width", "best_tile", "best_tile_speedup_vs_sw"]
    ]
)

print("\n=== Best tile vs image size ===")
display(df.groupby("height")["best_tile"].agg(lambda x: x.value_counts().index[0]))

print("\n=== Best tile distribution per size ===")
display(df.groupby("height")["best_tile"].value_counts())

print("\n=== Compute vs Overhead breakdown ===")
display(tile_df.groupby("tile_size")[["compute_ratio", "overhead_ratio"]].mean())

# =========================
# 5. FULL TABLES FOR REFERENCE
# =========================

print("\n=== Full Base Results ===")
display(df)

print("\n=== Full Tile Results ===")
display(tile_df)

In [ ]:
# Hardware vs Software Scaling (per kernel)
import matplotlib.pyplot as plt

for kernel in df["kernel"].unique():
    plt.figure(figsize=(8,5))

    subset = df[(df["kernel"] == kernel) & (df["sw_time"].notna())].sort_values("height")
    hw_subset = df[df["kernel"] == kernel].sort_values("height")

    plt.plot(subset["height"], subset["sw_time"], '--o', label="Software")
    plt.plot(hw_subset["height"], hw_subset["hw_total_time"], '-o', label="Hardware")

    plt.xlabel("Image Size (NxN)")
    plt.ylabel("Time (seconds)")
    plt.title(f"{kernel} - Software vs Hardware Scaling")

    plt.xscale("log", base=2)
    plt.yscale("log")
    plt.grid()
    plt.legend()

    plt.show()

In [ ]:
# Speedup vs Size (per kernel)
for kernel in df["kernel"].unique():
    plt.figure(figsize=(8,5))

    subset = df[(df["kernel"] == kernel) & (df["hw_speedup_vs_sw"].notna())]
    subset = subset.sort_values("height")

    plt.plot(subset["height"], subset["hw_speedup_vs_sw"], '-o', label=kernel)

    plt.axhline(1, linestyle='--', color='black', label="Baseline")
    plt.xscale("log", base=2)

    plt.xlabel("Image Size (NxN)")
    plt.ylabel("Speedup (SW / HW)")
    plt.title(f"{kernel} - Hardware Speedup vs Image Size")

    plt.grid()
    plt.legend()

    plt.show()

In [ ]:
# Tile Size vs Performance per kernel
for kernel in tile_df["kernel"].unique():
    plt.figure(figsize=(8,5))

    subset = tile_df[tile_df["kernel"] == kernel]
    grouped = subset.groupby("tile_size")["total_time"].mean()

    plt.plot(grouped.index, grouped.values, '-o')

    plt.xscale("log", base=2)
    plt.xlabel("Tile Size")
    plt.ylabel("Average Time (s)")
    plt.title(f"{kernel} - Tile Size vs Performance")

    plt.grid()
    plt.show()

In [ ]:
# Compute vs Overhead (per kernel)
for kernel in tile_df["kernel"].unique():
    plt.figure(figsize=(8,5))

    subset = tile_df[tile_df["kernel"] == kernel]
    grouped = subset.groupby("tile_size")[["compute_ratio", "overhead_ratio"]].mean()

    plt.plot(grouped.index, grouped["compute_ratio"], '-o', label="Compute")
    plt.plot(grouped.index, grouped["overhead_ratio"], '-o', label="Overhead")

    plt.xscale("log", base=2)
    plt.xlabel("Tile Size")
    plt.ylabel("Ratio")
    plt.title(f"{kernel} - Compute vs Overhead Tradeoff")

    plt.legend()
    plt.grid()
    plt.show()

In [ ]:
# Compute ratio (how much of the time is actual FPGA compute) across tile sizes
plt.figure(figsize=(8,5))

for kernel in tile_df["kernel"].unique():
    subset = tile_df[tile_df["kernel"] == kernel]
    grouped = subset.groupby("tile_size")["compute_ratio"].mean()

    plt.plot(grouped.index, grouped.values, '-o', label=kernel)

plt.xscale("log", base=2)
plt.xlabel("Tile Size")
plt.ylabel("Compute Ratio")
plt.title("Compute Efficiency Across Kernels")

plt.legend()
plt.grid()
plt.show()

In [ ]:
# Best Tile Distribution (per kernel)
for kernel in df["kernel"].unique():
    plt.figure(figsize=(6,4))

    subset = df[df["kernel"] == kernel]
    best_counts = subset["best_tile"].value_counts().sort_index()

    plt.bar(best_counts.index.astype(str), best_counts.values)

    plt.xlabel("Tile Size")
    plt.ylabel("Frequency")
    plt.title(f"{kernel} - Best Tile Distribution")

    plt.show()

In [ ]:
# Best Tile vs Image Size (per kernel)
for kernel in df["kernel"].unique():
    plt.figure(figsize=(8,5))

    subset = df[df["kernel"] == kernel]
    best_per_size = subset.groupby("height")["best_tile"].agg(lambda x: x.value_counts().index[0])

    plt.plot(best_per_size.index, best_per_size.values, '-o')

    plt.xscale("log", base=2)
    plt.xlabel("Image Size")
    plt.ylabel("Best Tile Size")
    plt.title(f"{kernel} — Optimal Tile Size vs Image Size")

    plt.grid()
    plt.show()

The hardware implementation includes data packing/unpacking and DMA transfers, which introduce non-trivial overhead. 
For smaller input sizes, this overhead dominates execution time, resulting in lower overall performance compared to the software baseline. 
As input size increases, the impact of this overhead diminishes, and the streaming architecture becomes more efficient. 
Due to its pipelined design (II = 1), the hardware achieves near-constant throughput, leading to improved performance and observable speedup at larger scales.